## 1. Install Dependencies

**Note**: If running on Colab, restart the runtime after installation.

In [2]:
!pip install -q -U \
    transformers datasets peft trl accelerate \
    bitsandbytes scikit-learn tqdm \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Optional: flash-attn for faster attention (requires CUDA 11.8+ and compatible GPU)
# !pip install flash-attn --no-build-isolation

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 160.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 195.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 161.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 32.7 MB/s 

## 2. Imports & Device Setup

In [3]:
# Cell 3 – Imports & constants

import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel  # if you want to use unsloth speedups

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

MODEL_NAME = "Trendyol/Trendyol-LLM-7b-chat-v0.1"          # ← corrected model id
DATASET_REPO = "yilmazzey/sdp2-nli"

MAX_SEQ_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5          # ← much safer than 2e-4 for classification
BATCH_SIZE = 2
GRAD_ACCUM = 8
NUM_EPOCHS = 1
WARMUP_RATIO = 0.1

print("Model:", MODEL_NAME)
print("Dataset repo:", DATASET_REPO)

/tmp/ipykernel_3046/918132026.py:8: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel  # if you want to use unsloth speedups


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Model: Trendyol/Trendyol-LLM-7b-chat-v0.1
Dataset repo: yilmazzey/sdp2-nli


## 3. Constants

In [4]:
REPO_ID = "yilmazzey/sdp2-nli"
CONFIGS = ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]
MODEL_ID = "Trendyol/Trendyol-LLM-7b-chat-v1.0"
NUM_LABELS = 3
LABEL_NAMES = ["entailment", "neutral", "contradiction"]

OUTPUT_DIR = "./trendyol-nli-finetuned"
RESULTS_DIR = "./results"
Path(RESULTS_DIR).mkdir(exist_ok=True)


NameError: name 'Path' is not defined

In [5]:
# Cell 4 – Load tokenizer & 4-bit model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,                     # auto
)

# Explicitly set pad_token to a string if it's not already defined
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Fix: Trendyol tokenizer stores some added-token "content" values as ints,
# which crashes tokenizer.save_pretrained() at checkpoint time.
from tokenizers import AddedToken as _AddedToken
for _idx, _tok in list(tokenizer.added_tokens_decoder.items()):
    if hasattr(_tok, 'content') and isinstance(_tok.content, int):
        tokenizer.added_tokens_decoder[_idx] = _AddedToken(
            str(_tok.content),
            single_word=_tok.single_word,
            lstrip=_tok.lstrip,
            rstrip=_tok.rstrip,
            normalized=_tok.normalized,
            special=_tok.special,
        )

tokenizer.padding_side = "right"

# Optional – enable unsloth 2× faster training + 60% less VRAM
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "v_proj"],   # ← cheaper & often better for NLI
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/718k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

Trendyol/Trendyol-LLM-7b-chat-v0.1 does not have a padding token! Will use pad_token = <unk>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [6]:
# Cell 5 – Prompt template & formatting function

def format_example(example):
    premise    = example["premise"]
    hypothesis = example["hypothesis"]

    # Map integer label → string label
    label_str = {0: "entailment", 1: "neutral", 2: "contradiction"}[example["label"]]

    # Very strict single-word answer format
    prompt = f"""[INST] Aşağıdaki önerme ve hipotez arasındaki mantıksal ilişkiyi belirle.
Sadece **tek kelime** cevap ver. Seçenekler: entailment, neutral, contradiction

Önerme: {premise}
Hipotez: {hypothesis}

Cevap: [/INST]"""

    full_text = prompt + label_str
    return {"text": full_text}


def formatting_func(example):
    return format_example(example)["text"]

In [7]:
# Cell 6 – Load & prepare datasets

print("Loading datasets...")

# Training = union of all train splits
train_parts = []
for name in ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]:
    ds = load_dataset(DATASET_REPO, name, split="train")
    train_parts.append(ds)

train_dataset = concatenate_datasets(train_parts).shuffle(seed=42)

# For quick debugging (remove or comment when you are ready for full run)
# train_dataset = train_dataset.select(range(25000))

print(f"Training examples: {len(train_dataset):,}")

# Evaluation – focus on native Turkish (TrGLUE)
eval_ds = load_dataset(DATASET_REPO, "trglue_mnli")

eval_sets = {
    "test_matched":   eval_ds["test_matched"],
    "test_mismatched": eval_ds["test_mismatched"],
}

for name, ds in eval_sets.items():
    ds = ds.map(format_example, remove_columns=ds.column_names)
    print(f"{name}: {len(ds):,} examples")

Loading datasets...


README.md: 0.00B [00:00, ?B/s]

snli_tr_1_1/train-00000-of-00001.parquet:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

snli_tr_1_1/validation-00000-of-00001.pa(…):   0%|          | 0.00/558k [00:00<?, ?B/s]

snli_tr_1_1/test-00000-of-00001.parquet:   0%|          | 0.00/557k [00:00<?, ?B/s]

...

Training examples: 1,103,874


Map:   0%|          | 0/9008 [00:00<?, ? examples/s]

test_matched: 9,008 examples


Map:   0%|          | 0/9217 [00:00<?, ? examples/s]

test_mismatched: 9,217 examples


In [9]:
# This is the function that formats + tokenizes each batch of examples
def tokenize_function(examples):
    # 1. Apply your formatting to get the full prompt + label strings
    texts = []
    for i in range(len(examples["premise"])):
        premise    = examples["premise"][i]
        hypothesis = examples["hypothesis"][i]
        label_str  = {0: "entailment", 1: "neutral", 2: "contradiction"}[examples["label"][i]]

        text = f"""[INST] Aşağıdaki önerme ve hipotez arasındaki mantıksal ilişkiyi belirle.
Sadece **tek kelime** cevap ver. Seçenekler: entailment, neutral, contradiction

Önerme: {premise}
Hipotez: {hypothesis}

Cevap: [/INST]{label_str}"""
        texts.append(text)

    # 2. Tokenize the formatted texts
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,               # we let data collator handle padding later
        return_tensors=None,         # return lists, not tensors
    )

    # For causal language modeling: labels are usually a copy of input_ids
    # (the trainer / data collator will handle shifting / masking)
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


# Apply it to the whole training dataset (this is where tokenized_train is born)
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,                    # faster when batched
    remove_columns=train_dataset.column_names,  # drop original columns
    desc="Tokenizing train",
    num_proc=4,                      # optional: speed up with multiprocessing
)


# You do the same for evaluation
tokenized_eval = eval_sets["test_matched"].map(
    tokenize_function,
    batched=True,
    remove_columns=eval_sets["test_matched"].column_names,
    desc="Tokenizing eval",
)

Tokenizing train (num_proc=4):   0%|          | 0/1103874 [00:00<?, ? examples/s]

Tokenizing eval:   0%|          | 0/9008 [00:00<?, ? examples/s]

In [10]:
from trl import SFTTrainer, SFTConfig

# If you haven't computed warmup_steps yet
total_steps = len(tokenized_train) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = SFTConfig(
    output_dir              = "./trendyol-nli-lora-v1",
    num_train_epochs        = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate           = LEARNING_RATE,
    warmup_steps            = warmup_steps,     # ← modern replacement (no deprecation warning)
    lr_scheduler_type       = "cosine",
    optim                   = "adamw_8bit",
    logging_steps           = 50,
    save_steps              = 1000,
    eval_strategy           = "steps",
    eval_steps              = 1000,
    save_total_limit        = 2,
    bf16                    = True,             # or False if GPU doesn't support
    max_grad_norm           = 0.3,
    weight_decay            = 0.01,
    report_to               = "none",

    # For pre-tokenized data → minimize processing
    max_seq_length          = MAX_SEQ_LENGTH,
    packing                 = False,
    # No dataset_text_field needed anymore (data is already tokenized)
    # No formatting_func needed
)

trainer = SFTTrainer(
    model           = model,                    # ← FIXED: no more ...
    tokenizer       = tokenizer,
    train_dataset   = tokenized_train,
    eval_dataset    = tokenized_eval,           # or None if you skip eval for now
    args            = training_args,
    # data_collator   = ... (optional: Unsloth usually handles it)
)


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
# (removed broken monkey-patch — fix is now in the tokenizer loading cell above)

In [13]:
# Cell 8 – Launch training

trainer.train()

# After training → merge & save (optional)
# model.save_pretrained_merged("trendyol-nli-merged", tokenizer, save_method="merged_16bit")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,103,874 | Num Epochs = 1 | Total steps = 68,993
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,388,608 of 6,846,926,848 (0.12% trained)


Step,Training Loss,Validation Loss
1000,1.468320,1.644272


TypeError: argument 'content': 'int' object cannot be converted to 'PyString'

In [ ]:
model.save_pretrained_merged("trendyol-nli-merged", tokenizer, save_method="merged_16bit")